In [ ]:
from google.colab import files

uploaded = files.upload()

Saving MaiaFormer-355-v0.1.zip to MaiaFormer-355-v0.1.zip


In [ ]:
import zipfile
from pathlib import Path

zip_path = "/content/MaiaFormer-355-v0.1.zip"
destino = Path("/content/maiaformer")

with zipfile.ZipFile(zip_path) as z:
    z.extractall(destino)

for arquivo in destino.rglob("*"):
    if arquivo.is_file():
        print(arquivo)

/content/maiaformer/maiaformer-355/maiaformer.py
/content/maiaformer/maiaformer-355/README.md
/content/maiaformer/maiaformer-355/config.json


In [ ]:
%cd /content/maiaformer/maiaformer-355

!python maiaformer.py --tiny

/content/maiaformer/maiaformer-355
Parameters: 328,320
Smoke test OK: torch.Size([2, 16, 256]) loss= 124.1833


In [ ]:
!python maiaformer.py --write-config config.json

Parameters: 351,456,896


In [ ]:
import torch
from maiaformer import Config, MaiaFormer

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

cfg = Config()

model = MaiaFormer(cfg)

print("Parâmetros:", sum(p.numel() for p in model.parameters()))

if torch.cuda.is_available():
    model = model.to("cuda")
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memória alocada (GB):",
          round(torch.cuda.memory_allocated() / 1024**3, 3))

PyTorch: 2.14.1+cu130
CUDA: True
Parâmetros: 351456896
GPU: Tesla T4
Memória alocada (GB): 1.315


In [ ]:
import torch
import time

model.eval()

device = next(model.parameters()).device

# Sequência sintética de 32 tokens
input_ids = torch.randint(
    0, cfg.vocab_size,
    (1, 32),
    device=device
)

torch.cuda.synchronize()
inicio = time.perf_counter()

with torch.inference_mode():
    logits = model(input_ids)

torch.cuda.synchronize()
tempo = time.perf_counter() - inicio

print("Formato dos logits:", logits.shape)
print("Tempo de forward:", round(tempo, 3), "s")
print("NaN:", torch.isnan(logits).any().item())
print("Inf:", torch.isinf(logits).any().item())
print("Média:", logits.float().mean().item())
print("Desvio padrão:", logits.float().std().item())

Formato dos logits: torch.Size([1, 32, 65536])
Tempo de forward: 0.776 s
NaN: False
Inf: False
Média: 0.004827363416552544
Desvio padrão: 30.04060935974121


In [ ]:
import torch
import torch.nn.functional as F

with torch.inference_mode():
    x = logits.float()

    print("Min:", x.min().item())
    print("Max:", x.max().item())
    print("Média:", x.mean().item())
    print("Desvio:", x.std().item())

    # Distribuição de probabilidade
    probs = F.softmax(x, dim=-1)

    # Entropia média em bits
    entropy = -(probs * probs.clamp_min(1e-30).log2()).sum(-1)

    print("Entropia média:", entropy.mean().item())
    print("Probabilidade máxima média:",
          probs.max(-1).values.mean().item())

    # Normas dos embeddings
    emb = model.get_input_embeddings().weight.float()

    print("Embedding std:", emb.std().item())
    print("Norma média embeddings:",
          emb.norm(dim=-1).mean().item())

Min: -154.68621826171875
Max: 674.4876708984375
Média: 0.004827363416552544
Desvio: 30.04060935974121
Entropia média: 0.0
Probabilidade máxima média: 1.0


AttributeError: 'MaiaFormer' object has no attribute 'get_input_embeddings'

In [ ]:
import torch

print("=== PARÂMETROS ===")

for name, p in model.named_parameters():
    if p.ndim >= 2 and (
        "embed" in name.lower()
        or "lm_head" in name.lower()
        or "token" in name.lower()
    ):
        w = p.detach().float()

        print(
            name,
            "shape:", tuple(w.shape),
            "std:", w.std().item(),
            "norma média:", w.norm(dim=-1).mean().item()
        )

print("\n=== TOP LOGITS ===")

top_values, top_ids = logits[0, -1].float().topk(5)

for token_id, value in zip(top_ids.tolist(), top_values.tolist()):
    print("Token:", token_id, "Logit:", value)

=== PARÂMETROS ===
model.embed_tokens.weight shape: (65536, 896) std: 0.9998555183410645 norma média: 29.920625686645508

=== TOP LOGITS ===
Token: 21262 Logit: 587.4705810546875
Token: 48006 Logit: 127.13062286376953
Token: 4845 Logit: 121.76583862304688
Token: 40421 Logit: 118.60154724121094
Token: 7655 Logit: 117.72484588623047


In [ ]:
import torch
import torch.nn as nn

# Reinicializar o modelo inteiro para descartar
# os pesos anteriores e evitar uma correção parcial.
del model

if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = MaiaFormer(cfg)

# Inicialização experimental para o protótipo.
# A política definitiva será auditada antes do treinamento.
with torch.no_grad():
    for module in model.modules():
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

model = model.to("cuda")
model.eval()

input_ids = torch.randint(
    0, cfg.vocab_size,
    (1, 32),
    device="cuda"
)

with torch.inference_mode():
    logits = model(input_ids).float()

    probs = torch.softmax(logits, dim=-1)

    entropy = -(
        probs * probs.clamp_min(1e-30).log2()
    ).sum(dim=-1)

print("=== NOVO DIAGNÓSTICO ===")
print("Logits min:", logits.min().item())
print("Logits max:", logits.max().item())
print("Logits std:", logits.std().item())
print("Entropia média:", entropy.mean().item())
print("Probabilidade máxima média:",
      probs.max(dim=-1).values.mean().item())

for name, p in model.named_parameters():
    if "embed_tokens.weight" in name:
        print("Embedding std:", p.float().std().item())
        print("Norma média:",
              p.float().norm(dim=-1).mean().item())

=== NOVO DIAGNÓSTICO ===
Logits min: -2.9671881198883057
Logits max: 2.7830777168273926
Logits std: 0.5996352434158325
Entropia média: 15.741015434265137
Probabilidade máxima média: 0.00016087305266410112
Embedding std: 0.020002007484436035
Norma média: 0.598557710647583


In [ ]:
print("=== MATRIZES DE EMBEDDING E SAÍDA ===")

for name, p in model.named_parameters():
    if "embed" in name.lower() or "head" in name.lower():
        print(name, tuple(p.shape))

print("\n=== MÓDULOS LINEARES ===")

for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        print(name, tuple(module.weight.shape))
        if name.count(".") > 2:
            break

=== MATRIZES DE EMBEDDING E SAÍDA ===
model.embed_tokens.weight (65536, 896)

=== MÓDULOS LINEARES ===
model.layers.0.self_attn.q_proj (896, 896)


In [ ]:
print("=== ESTRUTURA DO MODELO ===")

print(model)

print("\n=== PESOS COMPARTILHADOS ===")

for name, module in model.named_modules():
    if isinstance(module, torch.nn.Embedding):
        print("Embedding:", name, module.weight.shape)

    if isinstance(module, torch.nn.Linear) and (
        "lm_head" in name or "output" in name
    ):
        print("Saída:", name, module.weight.shape)

print("\n=== PARÂMETROS ===")
print("Total:", sum(p.numel() for p in model.parameters()))

=== ESTRUTURA DO MODELO ===
MaiaFormer(
  (model): Module(
    (embed_tokens): Embedding(65536, 896)
    (layers): ModuleList(
      (0-28): 29 x Layer(
        (self_attn): Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=False)
          (k_proj): Linear(in_features=896, out_features=128, bias=False)
          (v_proj): Linear(in_features=896, out_features=128, bias=False)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): MLP(
          (gate_proj): Linear(in_features=896, out_features=3072, bias=False)
          (up_proj): Linear(in_features=896, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=896, bias=False)
        )
        (input_layernorm): RMSNorm()
        (post_attention_layernorm): RMSNorm()
      )
    )
    (norm): RMSNorm()
  )
  (lm_head): Linear(in_features=896, out_features=65536, bias=False)
)

=== PESOS COMPARTILHADOS ===
Embedding: model.embed_

In [ ]:
emb = model.model.embed_tokens.weight
head = model.lm_head.weight

print("Mesmo objeto:", emb is head)
print("Mesmo armazenamento:", emb.data_ptr() == head.data_ptr())
print("Shape:", emb.shape)

Mesmo objeto: True
Mesmo armazenamento: True
Shape: torch.Size([65536, 896])


In [ ]:
print("=== CONFIGURAÇÃO ===")
print(cfg)

print("\n=== ATENÇÃO ===")
attn = model.model.layers[0].self_attn

for name in [
    "num_heads",
    "num_key_value_heads",
    "head_dim",
    "rope_theta",
]:
    print(name, getattr(attn, name, "não exposto"))

print("\n=== NORMALIZAÇÃO ===")
norm = model.model.layers[0].input_layernorm

for name in ["eps", "variance_epsilon"]:
    print(name, getattr(norm, name, "não exposto"))

=== CONFIGURAÇÃO ===
Config(vocab_size=65536, hidden_size=896, intermediate_size=3072, num_hidden_layers=29, num_attention_heads=14, num_key_value_heads=2, max_position_embeddings=2048, rms_norm_eps=1e-06, rope_theta=10000.0, tie_word_embeddings=True)

=== ATENÇÃO ===
num_heads não exposto
num_key_value_heads não exposto
head_dim 64
rope_theta não exposto

=== NORMALIZAÇÃO ===
eps 1e-06
variance_epsilon não exposto


In [ ]:
%pip install -q transformers safetensors

In [ ]:
import torch
import gc
from transformers import LlamaConfig, LlamaForCausalLM

hf_config = LlamaConfig(
    vocab_size=cfg.vocab_size,
    hidden_size=cfg.hidden_size,
    intermediate_size=cfg.intermediate_size,
    num_hidden_layers=cfg.num_hidden_layers,
    num_attention_heads=cfg.num_attention_heads,
    num_key_value_heads=cfg.num_key_value_heads,
    max_position_embeddings=cfg.max_position_embeddings,
    rms_norm_eps=cfg.rms_norm_eps,
    rope_theta=cfg.rope_theta,
    tie_word_embeddings=cfg.tie_word_embeddings,
    attention_bias=False,
    mlp_bias=False,
    attention_dropout=0.0,
)

# Preservar os pesos do MaiaFormer na CPU
maia_state = {
    k: v.detach().cpu().clone()
    for k, v in model.state_dict().items()
}

# Liberar memória da GPU
model = model.cpu()
gc.collect()
torch.cuda.empty_cache()

# Criar Llama na CPU para evitar pico de memória na T4
hf_model = LlamaForCausalLM(hf_config)

print("Parâmetros HF:",
      sum(p.numel() for p in hf_model.parameters()))

Parâmetros HF: 351456896


In [ ]:
result = hf_model.load_state_dict(
    maia_state,
    strict=True
)

print("Missing keys:", result.missing_keys)
print("Unexpected keys:", result.unexpected_keys)

Missing keys: []
Unexpected keys: []


In [ ]:
model.eval()
hf_model.eval()

# Comparar em CPU para reduzir diferenças entre kernels
device = "cpu"

model = model.to(device)
hf_model = hf_model.to(device)

torch.manual_seed(42)

input_ids = torch.randint(
    0,
    cfg.vocab_size,
    (1, 16),
    device=device
)

with torch.inference_mode():
    maia_logits = model(input_ids).float()

    hf_logits = hf_model(
        input_ids=input_ids,
        use_cache=False
    ).logits.float()

diff = (maia_logits - hf_logits).abs()

print("Shape Maia:", maia_logits.shape)
print("Shape HF:", hf_logits.shape)
print("Diferença máxima:", diff.max().item())
print("Diferença média:", diff.mean().item())

print(
    "Allclose:",
    torch.allclose(
        maia_logits,
        hf_logits,
        atol=1e-4,
        rtol=1e-4
    )
)

Shape Maia: torch.Size([1, 16, 65536])
Shape HF: torch.Size([1, 16, 65536])
Diferença máxima: 0.0
Diferença média: 0.0
Allclose: True


In [ ]:
from pathlib import Path

export_dir = Path("/content/maiaformer-355-hf")
export_dir.mkdir(parents=True, exist_ok=True)

hf_model.save_pretrained(
    export_dir,
    safe_serialization=True,
    max_shard_size="2GB"
)

print("Modelo exportado:", export_dir)

for path in export_dir.iterdir():
    print(path.name, round(path.stat().st_size / 1024**2, 2), "MB")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo exportado: /content/maiaformer-355-hf
model.safetensors 1340.73 MB
generation_config.json 0.0 MB
config.json 0.0 MB


In [ ]:
%pip install -q tokenizers transformers

In [ ]:
from tokenizers import Tokenizer, models, pre_tokenizers, decoders
from transformers import PreTrainedTokenizerFast
from pathlib import Path

export_dir = Path("/content/maiaformer-355-hf")

# Vocabulário sintético: apenas para testar a conversão
vocab = {
    f"<tok_{i}>": i
    for i in range(65536)
}

tokenizer_backend = Tokenizer(
    models.WordLevel(
        vocab=vocab,
        unk_token="<tok_0>"
    )
)

tokenizer_backend.pre_tokenizer = (
    pre_tokenizers.Whitespace()
)

tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer_backend,
    unk_token="<tok_0>",
    bos_token="<tok_1>",
    eos_token="<tok_2>",
    pad_token="<tok_3>",
)

tokenizer.save_pretrained(export_dir)

print("Tamanho do vocabulário:", len(tokenizer))
print("Arquivos:", [p.name for p in export_dir.iterdir()])

Tamanho do vocabulário: 65536
Arquivos: ['model.safetensors', 'tokenizer_config.json', 'generation_config.json', 'tokenizer.json', 'config.json']


In [ ]:
%cd /content

!git clone --depth 1 https://github.com/ggml-org/llama.cpp.git

/content
Cloning into 'llama.cpp'...
remote: Enumerating objects: 5081, done.
remote: Counting objects: 100% (5081/5081), done.
remote: Compressing objects: 100% (4233/4233), done.
remote: Total 5081 (delta 873), reused 3300 (delta 752), pack-reused 0 (from 0)
Receiving objects: 100% (5081/5081), 38.04 MiB | 8.56 MiB/s, done.
Resolving deltas: 100% (873/873), done.
Updating files: 100% (4619/4619), done.


In [ ]:
%cd /content/llama.cpp

%pip install -q -r requirements/requirements-convert_hf_to_gguf.txt

/content/llama.cpp
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
google-cloud-language 2.22.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
google-cloud-bigquery-storage 2.43.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is incompatible.
google-cloud-functions 1.25.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is incompatible.
google-cloud-spanner 3.71.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
google-clou

In [ ]:
!python convert_hf_to_gguf.py \
    /content/maiaformer-355-hf \
    --outfile /content/maiaformer-355-f32.gguf \
    --outtype f32

In [ ]:
from google.colab import drive
import shutil
from pathlib import Path

drive.mount("/content/drive")

origem = Path("/content/maiaformer-355-hf")
destino = Path(
    "/content/drive/MyDrive/MaiaFormer-355/maiaformer-355-hf"
)

destino.parent.mkdir(parents=True, exist_ok=True)

shutil.copytree(origem, destino, dirs_exist_ok=True)

print("Modelo preservado em:", destino)

Mounted at /content/drive
Modelo preservado em: /content/drive/MyDrive/MaiaFormer-355/maiaformer-355-hf


In [ ]:
%cd /content/llama.cpp

!python -m venv /content/gguf-env

!/content/gguf-env/bin/python -m pip install --upgrade pip

!/content/gguf-env/bin/python -m pip install \
    -r requirements/requirements-convert_hf_to_gguf.txt

/content/llama.cpp
Error: Command '['/content/gguf-env/bin/python3', '-m', 'ensurepip', '--upgrade', '--default-pip']' returned non-zero exit status 1.
/content/gguf-env/bin/python: No module named pip
/content/gguf-env/bin/python: No module named pip


In [ ]:
!/content/gguf-env/bin/python -c \
    "import torch; print('Torch:', torch.__version__); import gguf; print('GGUF: OK')"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import torch; print('Torch:', torch.__version__); import gguf; print('GGUF: OK')
    ^^^^^^^^^^^^
ModuleNotFoundError: No module named 'torch'


In [ ]:
!/content/gguf-env/bin/python -m pip install \
    -e /content/llama.cpp/gguf-py

/content/gguf-env/bin/python: No module named pip


In [ ]:
!/content/gguf-env/bin/python -m pip install \
    numpy safetensors sentencepiece protobuf \
    transformers tokenizers

/content/gguf-env/bin/python: No module named pip


In [ ]:
!/content/gguf-env/bin/python -c \
    "import gguf, numpy, safetensors, transformers; print('Dependências GGUF: OK')"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import gguf, numpy, safetensors, transformers; print('Dependências GGUF: OK')
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'gguf'


In [ ]:
import subprocess
from pathlib import Path

python = "/content/gguf-env/bin/python"
gguf_dir = Path("/content/llama.cpp/gguf-py")

print("=== PYTHON ===")
subprocess.run([python, "--version"])

print("\n=== DIRETÓRIO GGUF ===")
print("Existe:", gguf_dir.exists())
print("pyproject.toml:", (gguf_dir / "pyproject.toml").exists())

print("\n=== INSTALAÇÃO ===")
subprocess.run([
    python, "-m", "pip", "show", "gguf"
])

print("\n=== CAMINHOS PYTHON ===")
subprocess.run([
    python, "-c",
    "import sys; print('\\n'.join(sys.path))"
])

=== PYTHON ===

=== DIRETÓRIO GGUF ===
Existe: True
pyproject.toml: True

=== INSTALAÇÃO ===

=== CAMINHOS PYTHON ===


CompletedProcess(args=['/content/gguf-env/bin/python', '-c', "import sys; print('\\n'.join(sys.path))"], returncode=0)

In [38]:
import subprocess

PYTHON = "/content/gguf-env/bin/python"

# Tentar habilitar o pip pelo ensurepip
resultado = subprocess.run(
    [PYTHON, "-m", "ensurepip", "--upgrade"],
    capture_output=True,
    text=True
)

print(resultado.stdout)
print(resultado.stderr)
print("Return code:", resultado.returncode)


/content/gguf-env/bin/python: No module named ensurepip

Return code: 1


In [39]:
import subprocess
import sys

# Instalar virtualenv no ambiente principal
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "-q", "virtualenv"
], check=True)

# Criar ambiente isolado com pip
subprocess.run([
    sys.executable, "-m", "virtualenv",
    "/content/gguf-env2"
], check=True)

# Verificar
subprocess.run([
    "/content/gguf-env2/bin/python",
    "-m", "pip", "--version"
], check=True)

CompletedProcess(args=['/content/gguf-env2/bin/python', '-m', 'pip', '--version'], returncode=0)

In [40]:
import subprocess

PYTHON = "/content/gguf-env2/bin/python"

subprocess.run([
    PYTHON, "-m", "pip", "install",
    "-r",
    "/content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt"
], check=True)

subprocess.run([
    PYTHON, "-m", "pip", "install",
    "/content/llama.cpp/gguf-py"
], check=True)

subprocess.run([
    PYTHON, "-c",
    "import gguf, numpy, safetensors, transformers; "
    "print('Ambiente GGUF funcionando!')"
], check=True)

CompletedProcess(args=['/content/gguf-env2/bin/python', '-c', "import gguf, numpy, safetensors, transformers; print('Ambiente GGUF funcionando!')"], returncode=0)

In [43]:
import json
from pathlib import Path

export_dir = Path("/content/maiaformer-355-hf")
config_path = export_dir / "tokenizer_config.json"

config = json.loads(config_path.read_text())

print("Classe anterior:", config.get("tokenizer_class"))

config["tokenizer_class"] = "PreTrainedTokenizerFast"

config_path.write_text(
    json.dumps(config, indent=2, ensure_ascii=False)
)

print("Classe atual:", config["tokenizer_class"])

Classe anterior: TokenizersBackend
Classe atual: PreTrainedTokenizerFast


In [45]:
import subprocess

PYTHON = "/content/gguf-env2/bin/python"

codigo = """
from transformers import AutoTokenizer

path = "/content/maiaformer-355-hf"

tok = AutoTokenizer.from_pretrained(path)

print("Classe:", type(tok).__name__)
print("Vocabulário:", len(tok))
print("BOS:", tok.bos_token_id)
print("EOS:", tok.eos_token_id)

assert len(tok) == 65536
print("TOKENIZER HF: OK")
"""

subprocess.run(
    [PYTHON, "-c", codigo],
    check=True
)

CompletedProcess(args=['/content/gguf-env2/bin/python', '-c', '\nfrom transformers import AutoTokenizer\n\npath = "/content/maiaformer-355-hf"\n\ntok = AutoTokenizer.from_pretrained(path)\n\nprint("Classe:", type(tok).__name__)\nprint("Vocabulário:", len(tok))\nprint("BOS:", tok.bos_token_id)\nprint("EOS:", tok.eos_token_id)\n\nassert len(tok) == 65536\nprint("TOKENIZER HF: OK")\n'], returncode=0)

In [48]:
import subprocess
from pathlib import Path

PYTHON = "/content/gguf-env2/bin/python"

script = r'''
from transformers import AutoTokenizer
from pathlib import Path

export_dir = Path("/content/maiaformer-355-hf")

# Tokenizer GPT-2 BPE para teste de compatibilidade
tok = AutoTokenizer.from_pretrained(
    "gpt2",
    use_fast=True
)

print("Vocabulário original:", len(tok))

faltantes = 65536 - len(tok)

novos_tokens = [
    f"<maia_extra_{i}>"
    for i in range(faltantes)
]

adicionados = tok.add_tokens(novos_tokens)

print("Tokens adicionados:", adicionados)
print("Vocabulário final:", len(tok))

assert len(tok) == 65536

tok.save_pretrained(export_dir)

print("TOKENIZER BPE EXPERIMENTAL: OK")
'''

resultado = subprocess.run(
    [PYTHON, "-c", script],
    capture_output=True,
    text=True
)

print(resultado.stdout)
print(resultado.stderr)
print("Código de retorno:", resultado.returncode)

Vocabulário original: 50257
Tokens adicionados: 15279
Vocabulário final: 65536
TOKENIZER BPE EXPERIMENTAL: OK


Código de retorno: 0


In [50]:
import subprocess

PYTHON = "/content/gguf-env2/bin/python"

subprocess.run([
    PYTHON, "-c",
    """
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(
    '/content/maiaformer-355-hf',
    use_fast=True
)

print('Classe:', type(tok).__name__)
print('Vocabulário:', len(tok))
print('Modelo interno:', type(tok.backend_tokenizer.model).__name__)

assert len(tok) == 65536
"""
], check=True)

CompletedProcess(args=['/content/gguf-env2/bin/python', '-c', "\nfrom transformers import AutoTokenizer\n\ntok = AutoTokenizer.from_pretrained(\n    '/content/maiaformer-355-hf',\n    use_fast=True\n)\n\nprint('Classe:', type(tok).__name__)\nprint('Vocabulário:', len(tok))\nprint('Modelo interno:', type(tok.backend_tokenizer.model).__name__)\n\nassert len(tok) == 65536\n"], returncode=0)

In [49]:
import subprocess
from pathlib import Path

cmd = [
    "/content/gguf-env2/bin/python",
    "/content/llama.cpp/convert_hf_to_gguf.py",
    "/content/maiaformer-355-hf",
    "--outfile", "/content/maiaformer-355-f32.gguf",
    "--outtype", "f32"
]

log_path = Path("/content/maiaformer-gguf-conversion.log")

with log_path.open("w") as log:
    resultado = subprocess.run(
        cmd,
        stdout=log,
        stderr=subprocess.STDOUT
    )

print("Código de retorno:", resultado.returncode)
print("\n=== FINAL DO LOG ===")
print("\n".join(
    log_path.read_text(errors="replace").splitlines()[-50:]
))

gguf_path = Path("/content/maiaformer-355-f32.gguf")

if resultado.returncode == 0 and gguf_path.is_file():
    print("\nGGUF GERADO COM SUCESSO!")
    print("Tamanho:", round(gguf_path.stat().st_size / 1024**2, 2), "MiB")
else:
    print("\nConversão ainda não concluída.")

Código de retorno: 0

=== FINAL DO LOG ===
Writing:  34%|███▍      | 477M/1.41G [00:19<00:40, 22.8Mbyte/s]
Writing:  36%|███▌      | 499M/1.41G [00:20<00:36, 24.7Mbyte/s]
Writing:  36%|███▋      | 510M/1.41G [00:23<01:16, 11.8Mbyte/s]
Writing:  38%|███▊      | 540M/1.41G [00:23<00:43, 19.8Mbyte/s]
Writing:  40%|████      | 569M/1.41G [00:23<00:27, 30.2Mbyte/s]
Writing:  42%|████▏     | 591M/1.41G [00:26<00:47, 17.0Mbyte/s]
Writing:  43%|████▎     | 609M/1.41G [00:29<01:10, 11.3Mbyte/s]
Writing:  45%|████▍     | 631M/1.41G [00:29<00:49, 15.8Mbyte/s]
Writing:  47%|████▋     | 661M/1.41G [00:30<00:34, 21.6Mbyte/s]
Writing:  48%|████▊     | 672M/1.41G [00:30<00:33, 22.1Mbyte/s]
Writing:  48%|████▊     | 679M/1.41G [00:33<01:08, 10.6Mbyte/s]
Writing:  50%|████▉     | 701M/1.41G [00:34<00:43, 16.2Mbyte/s]
Writing:  51%|█████     | 719M/1.41G [00:34<00:30, 22.3Mbyte/s]
Writing:  53%|█████▎    | 741M/1.41G [00:34<00:21, 31.5Mbyte/s]
Writing:  54%|█████▍    | 756M/1.41G [00:35<00:33, 19.3Mbyte/

In [51]:
%cd /content/llama.cpp

!cmake -S . -B build -DCMAKE_BUILD_TYPE=Release
!cmake --build build --config Release -j 2

/content/llama.cpp
-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- llama.cpp version: 1.0.0-dev
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.43.0")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARC

In [52]:
from pathlib import Path

cli = Path("/content/llama.cpp/build/bin/llama-cli")
print("llama-cli disponível:", cli.exists())

llama-cli disponível: True


In [53]:
import subprocess

cmd = [
    "/content/llama.cpp/build/bin/llama-cli",
    "-m", "/content/maiaformer-355-f32.gguf",
    "-p", "Hello, Maia!",
    "-n", "16",
    "-c", "2048",
    "-t", "4",
    "--temp", "0"
]

resultado = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    timeout=300
)

print("Código de retorno:", resultado.returncode)
print(resultado.stdout[-12000:])

Código de retorno: 0


Loading model... |-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\ 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b1-1014775
model      : /content/maiaformer-355-f32.gguf
ftype      : all F32
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern



> Hello, Maia!
 Kimberly Kimberly Kimberly Kimberly Kimberly Kimberly Kimberly Kimberly Kimberly Kimberly Kimberly Kimberly Kimberly Kimberly Kimberly Kimberly

[ Prompt: 53.4 t/s | Generation: 5.8 t/s ]

> 

Exiting...



# MaiaFormer-355 — HF ↔ GGUF F32 numerical validation

**Goal:** compare the full `[sequence_length, 65536]` logit tensors produced by Hugging Face `LlamaForCausalLM` and the llama.cpp runtime (via `llama-cpp-python`). This is a numerical equivalence test, **not** a language-quality test.

**Requirements:** `/content/maiaformer-355-hf/` and `/content/maiaformer-355-f32.gguf` already exist. Run cells in order. This notebook does not change either model or tokenizer.

**Important:** The tokenizer in this exported checkpoint is temporary (GPT-2 extended to 65,536). Replace it before actual pretraining. This validation uses *explicit token IDs* to avoid differences in tokenization.


In [54]:
from pathlib import Path
HF_DIR = Path('/content/maiaformer-355-hf')
GGUF_PATH = Path('/content/maiaformer-355-f32.gguf')
assert (HF_DIR / 'config.json').is_file(), f'Missing: {HF_DIR}/config.json'
assert (HF_DIR / 'model.safetensors').is_file(), f'Missing: {HF_DIR}/model.safetensors'
assert GGUF_PATH.is_file(), f'Missing: {GGUF_PATH}'
print('HF:', HF_DIR)
print('GGUF:', GGUF_PATH, f'({GGUF_PATH.stat().st_size / 2**20:.1f} MiB)')


HF: /content/maiaformer-355-hf
GGUF: /content/maiaformer-355-f32.gguf (1342.8 MiB)


## 1. Isolated environment

The Colab main Python environment was previously affected by conflicting dependency versions. Create a **new** virtualenv for this validation. The llama.cpp Python binding compiles a CPU runtime; it may take several minutes. Install Torch CPU to avoid a large CUDA wheel. The notebook invokes the isolated interpreter through subprocess, without modifying the Colab kernel.


In [55]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'virtualenv'], check=True)
subprocess.run([sys.executable, '-m', 'virtualenv', '/content/maia-compare-env'], check=True)
PY = '/content/maia-compare-env/bin/python'
subprocess.run([PY, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True)
subprocess.run([PY, '-m', 'pip', 'install', '--index-url', 'https://download.pytorch.org/whl/cpu', 'torch'], check=True)
subprocess.run([PY, '-m', 'pip', 'install', 'transformers>=4.46,<5', 'safetensors', 'numpy<3', 'sentencepiece'], check=True)
subprocess.run([PY, '-m', 'pip', 'install', 'llama-cpp-python'], check=True)
print('Ready:', PY)


Ready: /content/maia-compare-env/bin/python


## 2. Full-logit comparison script

The C++ binding returns `scores` for all evaluated positions when `logits_all=True`. We disable BOS insertion and pass the same explicit IDs to both runtimes. We test short sequences to keep CPU execution manageable.

If the binding build fails, retain the error log and do not interpret it as an architecture failure. The C++ binding is a separate installation from the already-built `/content/llama.cpp` executable.


In [56]:
script = r'''import json, math, time
import numpy as np
import torch
from pathlib import Path
from transformers import LlamaForCausalLM
from llama_cpp import Llama

hf_dir = '/content/maiaformer-355-hf'
gguf_path = '/content/maiaformer-355-f32.gguf'
torch.set_num_threads(4)
print('Loading HF model (CPU, FP32)...', flush=True)
hf = LlamaForCausalLM.from_pretrained(hf_dir, torch_dtype=torch.float32, low_cpu_mem_usage=False)
hf.eval()
config = hf.config
print('HF config:', {'vocab_size': config.vocab_size, 'hidden_size': config.hidden_size,
 'num_hidden_layers': config.num_hidden_layers, 'num_attention_heads': config.num_attention_heads,
 'num_key_value_heads': config.num_key_value_heads}, flush=True)
print('Loading GGUF model (CPU, F32)...', flush=True)
llm = Llama(model_path=gguf_path, n_ctx=128, n_batch=32, n_gpu_layers=0,
            logits_all=True, embedding=False, verbose=False)

# Same token IDs in both engines. Avoid special-token assumptions and tokenizer differences.
sequences = {
    'short': [15496, 11, 995, 0],
    'mixed': [42, 1337, 1024, 32000, 50256, 65000, 7, 256],
    'repeated': [101, 101, 101, 101, 101, 101, 101, 101],
}
results = []
for name, ids in sequences.items():
    assert all(0 <= t < config.vocab_size for t in ids), (name, ids)
    print('\nSequence:', name, ids, flush=True)
    with torch.inference_mode():
        logits_hf = hf(input_ids=torch.tensor([ids], dtype=torch.long),
                       use_cache=False).logits[0].cpu().numpy().astype(np.float64)
    llm.reset()
    llm.eval(ids)
    logits_gguf = np.asarray(llm.scores, dtype=np.float64)
    # llama-cpp-python versions can allocate additional rows; only the last len(ids)
    # are relevant after reset() and eval(ids).
    assert logits_gguf.shape[0] >= len(ids), logits_gguf.shape
    logits_gguf = logits_gguf[:len(ids), :]
    assert logits_hf.shape == logits_gguf.shape, (logits_hf.shape, logits_gguf.shape)
    assert np.isfinite(logits_hf).all() and np.isfinite(logits_gguf).all(), 'Non-finite logits'
    diff = np.abs(logits_hf - logits_gguf)
    rel = diff / np.maximum(np.abs(logits_hf), 1e-3)
    top_hf = np.argmax(logits_hf, axis=-1)
    top_gguf = np.argmax(logits_gguf, axis=-1)
    # Stable softmax and KL divergence (HF || GGUF)
    def logsoftmax(x):
        z = x - x.max(axis=-1, keepdims=True)
        return z - np.log(np.exp(z).sum(axis=-1, keepdims=True))
    logp, logq = logsoftmax(logits_hf), logsoftmax(logits_gguf)
    kl = (np.exp(logp) * (logp - logq)).sum(axis=-1)
    row = {
      'sequence': name, 'positions': len(ids), 'vocab_size': logits_hf.shape[-1],
      'max_abs_error': float(diff.max()), 'mean_abs_error': float(diff.mean()),
      'p99_abs_error': float(np.percentile(diff, 99)),
      'max_relative_error_floor_1e-3': float(rel.max()),
      'mean_relative_error_floor_1e-3': float(rel.mean()),
      'argmax_agreement': float(np.mean(top_hf == top_gguf)),
      'hf_argmax': top_hf.tolist(), 'gguf_argmax': top_gguf.tolist(),
      'max_kl_hf_to_gguf_nats': float(np.max(np.maximum(kl, 0))),
      'mean_kl_hf_to_gguf_nats': float(np.mean(np.maximum(kl, 0))),
    }
    results.append(row)
    print(json.dumps(row, indent=2), flush=True)

out = {'model': 'MaiaFormer-355', 'format': 'GGUF F32', 'results': results}
Path('/content/maiaformer-hf-gguf-validation.json').write_text(json.dumps(out, indent=2))
print('\nSaved /content/maiaformer-hf-gguf-validation.json', flush=True)
'''
from pathlib import Path
Path('/content/compare_maiaformer_hf_gguf.py').write_text(script, encoding='utf-8')
print('Script created.')


Script created.


In [57]:
import subprocess
with open('/content/maiaformer-hf-gguf-validation.log', 'w') as log:
    p = subprocess.run([PY, '-u', '/content/compare_maiaformer_hf_gguf.py'],
                       stdout=log, stderr=subprocess.STDOUT, text=True)
print('Exit code:', p.returncode)
print(Path('/content/maiaformer-hf-gguf-validation.log').read_text()[-18000:])
if p.returncode != 0:
    print('Comparison did not finish. Share the log for diagnosis; no equivalence conclusion yet.')


Exit code: 0
Loading HF model (CPU, FP32)...
`torch_dtype` is deprecated! Use `dtype` instead!
HF config: {'vocab_size': 65536, 'hidden_size': 896, 'num_hidden_layers': 29, 'num_attention_heads': 14, 'num_key_value_heads': 2}
Loading GGUF model (CPU, F32)...

Sequence: short [15496, 11, 995, 0]
{
  "sequence": "short",
  "positions": 4,
  "vocab_size": 65536,
  "max_abs_error": 0.001520991325378418,
  "mean_abs_error": 0.00020120598968453862,
  "p99_abs_error": 0.0006555062532424931,
  "max_relative_error_floor_1e-3": 0.9114444255828857,
  "mean_relative_error_floor_1e-3": 0.0019953600295403596,
  "argmax_agreement": 1.0,
  "hf_argmax": [
    7606,
    14942,
    50691,
    55024
  ],
  "gguf_argmax": [
    7606,
    14942,
    50691,
    55024
  ],
  "max_kl_hf_to_gguf_nats": 3.78226522011395e-08,
  "mean_kl_hf_to_gguf_nats": 3.204119611194853e-08
}

Sequence: mixed [42, 1337, 1024, 32000, 50256, 65000, 7, 256]
{
  "sequence": "mixed",
  "positions": 8,
  "vocab_size": 65536,
  "max_a

## 3. How to interpret results

- **`argmax_agreement = 1.0`** for all positions is encouraging but **not sufficient** on its own.
- Examine **max/mean absolute errors**, the distribution of errors and **KL divergence**. Differences need not be zero because PyTorch and ggml use different CPU kernels and accumulation orders.
- If errors are large or the top token differs, investigate **tensor mapping, RoPE, RMSNorm, attention masking, tied embeddings and tokenizer metadata**. A single failing test does not identify which component is wrong.
- The short sequences test causal prefill. For complete parity, a subsequent test should compare **cached incremental decoding** as well as prefill.
- The Python binding may use a different llama.cpp commit than the separately compiled CLI. Record the versions before claiming parity for a specific commit.

**Share the output of the last cell**. We can then establish measured tolerances and decide whether the implementation passes.
